#### Convolution Layers와 PyTorch

- Conv1d(1차원 입력 데이터를 위한 Convolustion Layer, 일반적으로 Text-CNN에서 많이 사용)
- Conv2d(2차원 입력 데이터를 위한 Convolustion Layer, 일반적으로 이미지 분류에서 많이 사용)
- Conv3d(3차원 입력 데이터를 위한 Convolustion Layer)


#### Conv2d
Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0, dilation=1, groups=1, bias=True, padding_mode='zeros', device=None, dtype=None)

- in_channels (int): 입력 채널 수 (흑백 이미지일 경우는 1, RGB 값을 가진 이미지일 경우 3)
- out_channels (int): 출력 채널 수 
- kener_size (int or tuple): 커널 사이즈
- stride (int, tuple, optional): stride 사이즈 (Default : 1)
- padding (int, tuple or str, optional): padding 사이즈 (Default : 0)
- padding_mode (str, optional): 'zeros', 'reflect', 'replicate' or 'circular'. Default: 'zeros'
- dilation (int or tuple, optional): 커널 사이 간격 사이즈 (Default: 1)

In [44]:
import torch
import torch.nn as nn

conv1 = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3, padding=1)
print(conv1)
input1 = torch.Tensor(1, 1, 5, 5)
out1 = conv1(input1)
out1.shape

Conv2d(1, 1, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))


torch.Size([1, 1, 5, 5])

Pooling Layers
- 입력 데이터 차원에 맞추어, Max Pooling 또는 Average Pooling 을 적용할 수 있음

MaxPool2d
- MaxPool2d(kernel_size, stride=None, padding=0, dilation=1, return_indices=False, ceil_mode=False)

In [45]:
import torch
import torch.nn as nn

conv1 = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3, padding=1)
input1 = torch.Tensor(1, 1, 5, 5)
pool1 = nn.MaxPool2d(kernel_size=2)
out1 = conv1(input1)
out2 = pool1(out1)
print(out1.shape)
print(out2.shape)

torch.Size([1, 1, 5, 5])
torch.Size([1, 1, 2, 2])


#### 모델 정의
- Convolution Layer 는 입력 데이터에 필터(커널) 적용 후, activation 함수 적용한 Layer 의미함

1. Convolution Layer 는 입력 데이터에 필터(커널) 적용을 위한 전용 클래스 제공 (nn.Conv2d)
2. 이후에 Activation 함수 적용 (ex: nn.LearkyReLU(0.1))
3. 이후에 Batch Nomalization, Dropout 등 regularization 을 적용할 수도 있음 (옵션)
4. 이후에 Pooling 적용 (ex: nn.MaxPool2d)

In [46]:
conv1 = nn.Sequential(
    nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
    nn.LeakyReLU(0.1),
    nn.BatchNorm2d(32),
    nn.MaxPool2d(kernel_size=2, stride=2)
    # img = (?, 1, 28, 28)
    # Conv + Pool = (28 + 2 * 1 - 3) / 2 + 1 = 13 + 1 = 14, (?, 32, 14, 14)
)
input1 = torch.Tensor(1, 1, 28, 28)
out1 = conv1(input1)
out1.shape

torch.Size([1, 32, 14, 14])

In [47]:
conv1 = nn.Sequential(
    nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
    nn.LeakyReLU(0.1),
    nn.BatchNorm2d(32), # 출력 채널 수를 입력으로 받음
    nn.MaxPool2d(2),
    # img = (?, 1, 28, 28)
    # Conv = (28 + 2 * 1 - 3) / 1 + 1 = 28, (1, 32, 28, 28)
    # MaxPool = (28 - 2) / 2 = 14, (?, 32, 14, 14)
    nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
    nn.LeakyReLU(0.1),
    nn.BatchNorm2d(64),
    nn.MaxPool2d(kernel_size=2),
    # Conv = (14 + 2 * 1 - 3) + 1 = 13 + 1 = 14, (32, 64, 14, 14)
    # MaxPool = (14 - 2) / 2 = 7, (32, 64, 7, 7)
    nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
    nn.LeakyReLU(0.1),
    nn.BatchNorm2d(128),
    nn.MaxPool2d(2)
    # Conv = (7 + (2 * 1) - 3) + 1 = 7, (64, 128, 7, 7)
    # MaxPool = 7 / 2 = 3, (64, 128, 3, 3)
)

input1 = torch.Tensor(1, 1, 28, 28)
out1 = conv1(input1)
out2 = out1.view(out1.size(0), -1) # flatten
print(out1.shape, out2.shape, 128 * 3 * 3)

torch.Size([1, 128, 3, 3]) torch.Size([1, 1152]) 1152


#### CNN 모델 구성
1. 다음 세트로 하나의 Convolution Layer + Polling Layer를 구성하고, 여러 세트로 구축


In [48]:
class CNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            nn.LeakyReLU(0.1),
            nn.BatchNorm2d(32), # 출력 채널 수를 입력으로 받음
            nn.MaxPool2d(2),
            # img = (?, 1, 28, 28)
            # Conv = (28 + 2 * 1 - 3) / 1 + 1 = 28, (1, 32, 28, 28)
            # MaxPool = 28 / 2 = 14, (?, 32, 14, 14)
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.LeakyReLU(0.1),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(kernel_size=2),
            # Conv = (14 + 2 * 1 - 3) + 1 = 13 + 1 = 14, (32, 64, 14, 14)
            # MaxPool = 14 / 2 = 7, (32, 64, 7, 7)
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.LeakyReLU(0.1),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(2)
            # Conv = (7 + (2 * 1) - 3) + 1 = 7, (64, 128, 7, 7)
            # MaxPool = 7 / 2 = 3, (64, 128, 3, 3)
        )
        self.linear_layers = nn.Sequential(
            nn.Linear(128 * 3 * 3, 128),
            nn.LeakyReLU(0.1),
            nn.BatchNorm1d(128), # Linear Layer 이므로, BatchNorm1d 사용
            nn.Linear(128, 64),
            nn.LeakyReLU(0.1),
            nn.BatchNorm1d(64), # Linear Layer 이므로, BatchNorm1d 사용
            nn.Linear(64, 10),
            nn.LogSoftmax(dim=-1)
        )
    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # flatten
        x = self.linear_layers(x)
        return x

**MNIST with CNN**

In [49]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
import numpy as np
from copy import deepcopy
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


In [50]:
train_rowdata = datasets.MNIST('./dataset', train=True, download=True, transform=transforms.ToTensor())
test_dataset = datasets.MNIST('./dataset', train=False, download=True, transform=transforms.ToTensor())

In [69]:
train_idx, valid_idx, _, _ = train_test_split(list(range(len(train_rowdata))), 
                                             train_rowdata.targets, 
                                             test_size=0.2,
                                             stratify=train_rowdata.targets
                                             )

In [104]:
train_dataset = Subset(train_rowdata, train_idx)
validation_dataset = Subset(train_rowdata, valid_idx)

In [105]:
print(len(train_dataset), len(validation_dataset), len(test_dataset))

48000 12000 10000


In [106]:
minibatch_size = 128

train_batches = DataLoader(train_dataset, batch_size=minibatch_size, shuffle=True)
validation_batches = DataLoader(validation_dataset,batch_size=minibatch_size, shuffle=True)
test_batches = DataLoader(test_dataset, batch_size=minibatch_size, shuffle=False)


#### CNNModel 객체 생성

In [107]:
model = CNNModel().to(device)
model


CNNModel(
  (conv_layers): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): LeakyReLU(negative_slope=0.1)
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): LeakyReLU(negative_slope=0.1)
    (6): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): LeakyReLU(negative_slope=0.1)
    (10): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (linear_layers): Sequential(
    (0): Linear(in_features=1152, out_features=128, bias=True)

#### input, output, loss, optimizer 설정

In [108]:
loss_func = nn.NLLLoss() # log softmax 는 NLLLoss() 로 진행해야 함
optimizer = torch.optim.Adam(model.parameters())

#### Training * Validation

In [109]:
def train_model(model, early_stop, n_epochs, process_interval, device):
    train_losses, valid_losses, lowest_loss = list(), list(), np.inf
    model = model.to(device)

    for epoch in range(n_epochs):
        train_loss, validation_loss = 0.0, 0.0
        train_steps, validation_steps = 0, 0

        # ==== Train ====
        model.train()
        for x_minibatch, y_minibatch in train_batches:
            x_minibatch = x_minibatch.to(device)
            y_minibatch = y_minibatch.to(device)

            y_minibatch_pred = model(x_minibatch)
            loss = loss_func(y_minibatch_pred, y_minibatch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())
            train_loss += loss.item()
            train_steps += 1

        if train_steps:
            train_loss /= train_steps

        # ==== Validation ====
        model.eval()
        with torch.no_grad():
            for x_minibatch, y_minibatch in validation_batches:
                x_minibatch = x_minibatch.to(device)
                y_minibatch = y_minibatch.to(device)

                y_minibatch_pred = model(x_minibatch)
                loss = loss_func(y_minibatch_pred, y_minibatch)

                valid_losses.append(loss.item())
                validation_loss += loss.item()
                validation_steps += 1

        if validation_steps:
            validation_loss /= validation_steps

        current_valid_loss = valid_losses[-1]
        if current_valid_loss < lowest_loss:
            lowest_loss = current_valid_loss
            lowest_epoch = epoch
            best_model = deepcopy({k: v.detach().cpu() for k, v in model.state_dict().items()})
        else:
            if early_stop > 0 and lowest_epoch + early_stop < epoch:
                print('Early Stopped', epoch, 'epochs')
                model.load_state_dict(best_model)
                model = model.to(device)
                break

        if epoch % process_interval == 0:
            print(f"[{epoch:03d}] train_loss={train_loss:.4f}, val_loss={validation_loss:.4f}, best={lowest_loss:.4f}")

    model.load_state_dict(best_model)
    model = model.to(device)
    return model, lowest_loss, train_losses, valid_losses


훈련 실행

In [110]:
nb_epochs = 30
process_interval = 3
early_stop = 10

model, lowest_loss, train_losses, valid_losses = train_model(model, early_stop, nb_epochs, process_interval, device)


[000] train_loss=0.0000, val_loss=0.0000, best=0.0165
[003] train_loss=0.0000, val_loss=0.0000, best=0.0020
[006] train_loss=0.0000, val_loss=0.0000, best=0.0004
[009] train_loss=0.0000, val_loss=0.0000, best=0.0004
[012] train_loss=0.0000, val_loss=0.0000, best=0.0004
[015] train_loss=0.0000, val_loss=0.0000, best=0.0004
[018] train_loss=0.0000, val_loss=0.0000, best=0.0004
Early Stopped 21 epochs


In [ ]:
import torch

print(torch.backends.mps.is_available())